# 03 — Multi-Qubit Systems

This module introduces quantum systems containing
multiple qubits.

## Topics Covered

- Two-qubit systems
- Computational basis states
- Tensor products
- Multi-qubit superposition
- Measurement
- CNOT
- Bell states
- Entanglement
- Density matrices
- Partial trace
- Entangled vs separable states
- Three-qubit systems
- GHZ states

### Learning Philosophy

**Understand → Implement → Experiment → Observe → Analyze**

## 1. Import Libraries

We will use Qiskit to create and simulate quantum circuits.

Important classes:

- `QuantumCircuit` — creates quantum circuits
- `AerSimulator` — simulates circuits
- `Statevector` — represents quantum states
- `DensityMatrix` — represents mixed or reduced states
- `partial_trace` — obtains reduced quantum states
- `plot_histogram` — visualizes measurement results

In [1]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.quantum_info import (
    Statevector,
    DensityMatrix,
    partial_trace
)
from qiskit.visualization import plot_histogram

import numpy as np
import matplotlib.pyplot as plt

print("Libraries imported successfully!")

Libraries imported successfully!


## 2. What Is a Multi-Qubit System?

A multi-qubit system contains two or more qubits.

For example:

- 1 qubit → 2 basis states
- 2 qubits → 4 basis states
- 3 qubits → 8 basis states
- n qubits → 2ⁿ basis states

For two qubits, the computational basis is:

- `|00⟩`
- `|01⟩`
- `|10⟩`
- `|11⟩`

In [2]:
num_qubits = 2
num_states = 2 ** num_qubits

print("Number of qubits:", num_qubits)
print("Number of basis states:", num_states)

Number of qubits: 2
Number of basis states: 4


## 3. Creating a Two-Qubit Circuit

Let's create a circuit containing two qubits.

Initially both qubits are in:

`|00⟩`

In [3]:
qc = QuantumCircuit(2)

print(qc)

     
q_0: 
     
q_1: 
     


In [4]:
state = Statevector.from_instruction(qc)

print("Initial state:")
print(state)

Initial state:
Statevector([1.+0.j, 0.+0.j, 0.+0.j, 0.+0.j],
            dims=(2, 2))


## 4. Computational Basis States

A two-qubit system has four computational basis states:

`|00⟩`, `|01⟩`, `|10⟩`, and `|11⟩`.

We can create each state using X gates.

In [5]:
# |00>
qc_00 = QuantumCircuit(2)
state_00 = Statevector.from_instruction(qc_00)

print("|00>:", state_00)

|00>: Statevector([1.+0.j, 0.+0.j, 0.+0.j, 0.+0.j],
            dims=(2, 2))


In [6]:
# |01>
qc_01 = QuantumCircuit(2)
qc_01.x(0)

state_01 = Statevector.from_instruction(qc_01)

print("|01>:", state_01)

|01>: Statevector([0.+0.j, 1.+0.j, 0.+0.j, 0.+0.j],
            dims=(2, 2))


In [7]:
# |10>
qc_10 = QuantumCircuit(2)
qc_10.x(1)

state_10 = Statevector.from_instruction(qc_10)

print("|10>:", state_10)

|10>: Statevector([0.+0.j, 0.+0.j, 1.+0.j, 0.+0.j],
            dims=(2, 2))


In [8]:
# |11>
qc_11 = QuantumCircuit(2)
qc_11.x(0)
qc_11.x(1)

state_11 = Statevector.from_instruction(qc_11)

print("|11>:", state_11)

|11>: Statevector([0.+0.j, 0.+0.j, 0.+0.j, 1.+0.j],
            dims=(2, 2))


## 5. Tensor Products

Multiple-qubit states are constructed using tensor products.

For example:

`|0⟩ ⊗ |1⟩ = |01⟩`

The tensor product combines individual quantum systems
into one larger quantum system.

In [9]:
qubit_0 = Statevector.from_label("0")
qubit_1 = Statevector.from_label("1")

combined_state = qubit_0.tensor(qubit_1)

print("Qubit 0:")
print(qubit_0)

print("\nQubit 1:")
print(qubit_1)

print("\nTensor product |0> ⊗ |1>:")
print(combined_state)

Qubit 0:
Statevector([1.+0.j, 0.+0.j],
            dims=(2,))

Qubit 1:
Statevector([0.+0.j, 1.+0.j],
            dims=(2,))

Tensor product |0> ⊗ |1>:
Statevector([0.+0.j, 1.+0.j, 0.+0.j, 0.+0.j],
            dims=(2, 2))


## 6. Two-Qubit Superposition

Applying a Hadamard gate to each qubit creates:

`|++⟩`

The resulting state is:

`1/2 (|00⟩ + |01⟩ + |10⟩ + |11⟩)`

Therefore each computational basis state has probability:

**25%**

In [10]:
qc_superposition = QuantumCircuit(2)

qc_superposition.h(0)
qc_superposition.h(1)

print(qc_superposition)

     ┌───┐
q_0: ┤ H ├
     ├───┤
q_1: ┤ H ├
     └───┘


In [11]:
superposition_state = Statevector.from_instruction(
    qc_superposition
)

print("Statevector:")
print(superposition_state)

print("\nProbabilities:")
print(superposition_state.probabilities_dict())

Statevector:
Statevector([0.5+0.j, 0.5+0.j, 0.5+0.j, 0.5+0.j],
            dims=(2, 2))

Probabilities:
{np.str_('00'): np.float64(0.2499999999999999), np.str_('01'): np.float64(0.2499999999999999), np.str_('10'): np.float64(0.2499999999999999), np.str_('11'): np.float64(0.2499999999999999)}


## 7. Measuring Two Qubits

Measurement converts a quantum state into classical
information.

We will measure the two-qubit superposition using
1024 shots.

The counts should be approximately equal.

In [12]:
measurement_circuit = QuantumCircuit(2, 2)

measurement_circuit.h(0)
measurement_circuit.h(1)

measurement_circuit.measure(
    [0, 1],
    [0, 1]
)

print(measurement_circuit)

     ┌───┐┌─┐   
q_0: ┤ H ├┤M├───
     ├───┤└╥┘┌─┐
q_1: ┤ H ├─╫─┤M├
     └───┘ ║ └╥┘
c: 2/══════╩══╩═
           0  1 


In [13]:
simulator = AerSimulator()

result = simulator.run(
    measurement_circuit,
    shots=1024
).result()

counts = result.get_counts()

print("Measurement results:")
print(counts)

Measurement results:
{'00': 277, '10': 233, '01': 254, '11': 260}


In [14]:
plot_histogram(counts)
plt.show()

## 8. CNOT — Controlled-X Gate

The CNOT gate has:

- One control qubit
- One target qubit

The target qubit flips only when the control qubit is `|1⟩`.

In Qiskit:

```python
qc.cx(control, target)
```

### Important Qiskit Bit Ordering

Qiskit displays classical bitstrings with the highest-indexed
bit on the left. Therefore printed bitstrings can look reversed
compared with the order in which qubits are written.

In [15]:
qc_cnot = QuantumCircuit(2)

qc_cnot.x(0)
qc_cnot.cx(0, 1)

print(qc_cnot)

     ┌───┐     
q_0: ┤ X ├──■──
     └───┘┌─┴─┐
q_1: ─────┤ X ├
          └───┘


In [16]:
cnot_state = Statevector.from_instruction(qc_cnot)

print("Final state:")
print(cnot_state)

print("\nProbabilities:")
print(cnot_state.probabilities_dict())

Final state:
Statevector([0.+0.j, 0.+0.j, 0.+0.j, 1.+0.j],
            dims=(2, 2))

Probabilities:
{np.str_('11'): np.float64(1.0)}


## 9. CNOT Truth Table

For a CNOT gate:

`|a,b⟩ → |a, a XOR b⟩`

The first qubit acts as the control.

In [17]:
def test_cnot(input_state):
    qc = QuantumCircuit(2)

    if input_state[0] == "1":
        qc.x(0)

    if input_state[1] == "1":
        qc.x(1)

    qc.cx(0, 1)

    state = Statevector.from_instruction(qc)

    return state.probabilities_dict()


inputs = ["00", "01", "10", "11"]

for state in inputs:
    result = test_cnot(state)
    print(f"Input |{state}> -> {result}")

Input |00> -> {np.str_('00'): np.float64(1.0)}
Input |01> -> {np.str_('10'): np.float64(1.0)}
Input |10> -> {np.str_('11'): np.float64(1.0)}
Input |11> -> {np.str_('01'): np.float64(1.0)}


## 10. Bell State — |Φ⁺⟩

Bell states are maximally entangled two-qubit states.

The first Bell state is:

`|Φ⁺⟩ = 1/√2 (|00⟩ + |11⟩)`

We create it using:

1. Hadamard on qubit 0
2. CNOT with qubit 0 as control and qubit 1 as target

In [18]:
bell_circuit = QuantumCircuit(2)

bell_circuit.h(0)
bell_circuit.cx(0, 1)

print(bell_circuit)

     ┌───┐     
q_0: ┤ H ├──■──
     └───┘┌─┴─┐
q_1: ─────┤ X ├
          └───┘


In [19]:
bell_state = Statevector.from_instruction(
    bell_circuit
)

print("Bell state:")
print(bell_state)

Bell state:
Statevector([0.70710678+0.j, 0.        +0.j, 0.        +0.j,
             0.70710678+0.j],
            dims=(2, 2))


In [20]:
print("Bell-state probabilities:")

bell_probabilities = bell_state.probabilities_dict()

print(bell_probabilities)

Bell-state probabilities:
{np.str_('00'): np.float64(0.4999999999999999), np.str_('11'): np.float64(0.4999999999999999)}


## 11. Measuring the Bell State

The Bell state contains only `|00⟩` and `|11⟩`.

Each should occur with approximately 50% probability.

In [21]:
bell_measurement = QuantumCircuit(2, 2)

bell_measurement.h(0)
bell_measurement.cx(0, 1)

bell_measurement.measure(
    [0, 1],
    [0, 1]
)

print(bell_measurement)

     ┌───┐     ┌─┐   
q_0: ┤ H ├──■──┤M├───
     └───┘┌─┴─┐└╥┘┌─┐
q_1: ─────┤ X ├─╫─┤M├
          └───┘ ║ └╥┘
c: 2/═══════════╩══╩═
                0  1 


In [22]:
bell_result = simulator.run(
    bell_measurement,
    shots=1024
).result()

bell_counts = bell_result.get_counts()

print("Bell-state measurement:")
print(bell_counts)

Bell-state measurement:
{'11': 527, '00': 497}


In [23]:
plot_histogram(bell_counts)
plt.show()

## 12. The Four Bell States

There are four standard Bell states:

### |Φ⁺⟩
`1/√2 (|00⟩ + |11⟩)`

### |Φ⁻⟩
`1/√2 (|00⟩ − |11⟩)`

### |Ψ⁺⟩
`1/√2 (|01⟩ + |10⟩)`

### |Ψ⁻⟩
`1/√2 (|01⟩ − |10⟩)`

In [24]:
# |Phi+>
phi_plus = QuantumCircuit(2)
phi_plus.h(0)
phi_plus.cx(0, 1)

# |Phi->
phi_minus = QuantumCircuit(2)
phi_minus.h(0)
phi_minus.cx(0, 1)
phi_minus.z(0)

# |Psi+>
psi_plus = QuantumCircuit(2)
psi_plus.h(0)
psi_plus.cx(0, 1)
psi_plus.x(1)

# |Psi->
psi_minus = QuantumCircuit(2)
psi_minus.h(0)
psi_minus.cx(0, 1)
psi_minus.x(1)
psi_minus.z(0)

bell_circuits = {
    "Phi+": phi_plus,
    "Phi-": phi_minus,
    "Psi+": psi_plus,
    "Psi-": psi_minus
}

for name, circuit in bell_circuits.items():
    state = Statevector.from_instruction(circuit)

    print(f"\n{name}:")
    print(state)


Phi+:
Statevector([0.70710678+0.j, 0.        +0.j, 0.        +0.j,
             0.70710678+0.j],
            dims=(2, 2))

Phi-:
Statevector([ 0.70710678+0.j, -0.        +0.j,  0.        +0.j,
             -0.70710678+0.j],
            dims=(2, 2))

Psi+:
Statevector([0.        +0.j, 0.70710678+0.j, 0.70710678+0.j,
             0.        +0.j],
            dims=(2, 2))

Psi-:
Statevector([ 0.        +0.j, -0.70710678+0.j,  0.70710678+0.j,
             -0.        +0.j],
            dims=(2, 2))


## 13. Understanding Entanglement

For an entangled system, the complete state cannot be
represented as independent states of the individual qubits.

Example:

`|Φ⁺⟩ = 1/√2 (|00⟩ + |11⟩)`

The measurement results are strongly correlated.

In [25]:
print("Bell state:")
print(bell_state)

print("\nProbabilities:")
print(bell_state.probabilities_dict())

Bell state:
Statevector([0.70710678+0.j, 0.        +0.j, 0.        +0.j,
             0.70710678+0.j],
            dims=(2, 2))

Probabilities:
{np.str_('00'): np.float64(0.4999999999999999), np.str_('11'): np.float64(0.4999999999999999)}


## 14. Density Matrix

For a pure state:

`ρ = |ψ⟩⟨ψ|`

We can convert the Bell state into a density matrix.

In [26]:
density_matrix = DensityMatrix(bell_state)

print("Bell-state density matrix:")
print(density_matrix)

Bell-state density matrix:
DensityMatrix([[0.5+0.j, 0. +0.j, 0. +0.j, 0.5+0.j],
               [0. +0.j, 0. +0.j, 0. +0.j, 0. +0.j],
               [0. +0.j, 0. +0.j, 0. +0.j, 0. +0.j],
               [0.5+0.j, 0. +0.j, 0. +0.j, 0.5+0.j]],
              dims=(2, 2))


## 15. Partial Trace

The partial trace obtains the reduced state of one part
of a larger quantum system.

For the Bell state, tracing out one qubit leaves the other
qubit in a mixed reduced state.

In [27]:
# Trace out qubit 1
reduced_qubit_0 = partial_trace(
    density_matrix,
    [1]
)

# Trace out qubit 0
reduced_qubit_1 = partial_trace(
    density_matrix,
    [0]
)

print("Reduced state of qubit 0:")
print(reduced_qubit_0)

print("\nReduced state of qubit 1:")
print(reduced_qubit_1)

Reduced state of qubit 0:
DensityMatrix([[0.5+0.j, 0. +0.j],
               [0. +0.j, 0.5+0.j]],
              dims=(2,))

Reduced state of qubit 1:
DensityMatrix([[0.5+0.j, 0. +0.j],
               [0. +0.j, 0.5+0.j]],
              dims=(2,))


## 16. Entangled vs Separable States

A separable state can be written as a tensor product
of individual states.

Example:

`|00⟩ = |0⟩ ⊗ |0⟩`

An entangled state cannot be written this way.

Example:

`|Φ⁺⟩ = 1/√2 (|00⟩ + |11⟩)`

In [28]:
# Separable state
separable_circuit = QuantumCircuit(2)

separable_state = Statevector.from_instruction(
    separable_circuit
)

print("Separable state:")
print(separable_state)

# Entangled state
entangled_circuit = QuantumCircuit(2)

entangled_circuit.h(0)
entangled_circuit.cx(0, 1)

entangled_state = Statevector.from_instruction(
    entangled_circuit
)

print("\nEntangled state:")
print(entangled_state)

Separable state:
Statevector([1.+0.j, 0.+0.j, 0.+0.j, 0.+0.j],
            dims=(2, 2))

Entangled state:
Statevector([0.70710678+0.j, 0.        +0.j, 0.        +0.j,
             0.70710678+0.j],
            dims=(2, 2))


## 17. Three-Qubit System

With three qubits:

`2³ = 8`

computational basis states are possible.

They range from `|000⟩` to `|111⟩`.

In [29]:
three_qubit = QuantumCircuit(3)

three_state = Statevector.from_instruction(
    three_qubit
)

print("Three-qubit initial state:")
print(three_state)

print("\nNumber of basis states:")
print(2 ** 3)

Three-qubit initial state:
Statevector([1.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j,
             0.+0.j],
            dims=(2, 2, 2))

Number of basis states:
8


## 18. Three-Qubit Superposition

Applying Hadamard gates to all three qubits creates
an equal superposition of all eight basis states.

Each has probability:

`1/8 = 12.5%`

In [30]:
three_superposition = QuantumCircuit(3)

three_superposition.h(0)
three_superposition.h(1)
three_superposition.h(2)

print(three_superposition)

     ┌───┐
q_0: ┤ H ├
     ├───┤
q_1: ┤ H ├
     ├───┤
q_2: ┤ H ├
     └───┘


In [31]:
three_state = Statevector.from_instruction(
    three_superposition
)

print("Statevector:")
print(three_state)

print("\nProbabilities:")
print(three_state.probabilities_dict())

Statevector:
Statevector([0.35355339+0.j, 0.35355339+0.j, 0.35355339+0.j,
             0.35355339+0.j, 0.35355339+0.j, 0.35355339+0.j,
             0.35355339+0.j, 0.35355339+0.j],
            dims=(2, 2, 2))

Probabilities:
{np.str_('000'): np.float64(0.12499999999999994), np.str_('001'): np.float64(0.12499999999999994), np.str_('010'): np.float64(0.12499999999999994), np.str_('011'): np.float64(0.12499999999999994), np.str_('100'): np.float64(0.12499999999999994), np.str_('101'): np.float64(0.12499999999999994), np.str_('110'): np.float64(0.12499999999999994), np.str_('111'): np.float64(0.12499999999999994)}


## 19. Measuring Three Qubits

In [32]:
three_measurement = QuantumCircuit(3, 3)

three_measurement.h(0)
three_measurement.h(1)
three_measurement.h(2)

three_measurement.measure(
    [0, 1, 2],
    [0, 1, 2]
)

print(three_measurement)

     ┌───┐┌─┐      
q_0: ┤ H ├┤M├──────
     ├───┤└╥┘┌─┐   
q_1: ┤ H ├─╫─┤M├───
     ├───┤ ║ └╥┘┌─┐
q_2: ┤ H ├─╫──╫─┤M├
     └───┘ ║  ║ └╥┘
c: 3/══════╩══╩══╩═
           0  1  2 


In [33]:
three_result = simulator.run(
    three_measurement,
    shots=1024
).result()

three_counts = three_result.get_counts()

print("Three-qubit measurement:")
print(three_counts)

Three-qubit measurement:
{'101': 144, '100': 121, '010': 124, '000': 128, '011': 134, '111': 132, '110': 119, '001': 122}


In [34]:
plot_histogram(three_counts)
plt.show()

## 20. GHZ State

The three-qubit GHZ state is:

`|GHZ⟩ = 1/√2 (|000⟩ + |111⟩)`

We create it using:

1. Hadamard on qubit 0
2. CNOT from qubit 0 to qubit 1
3. CNOT from qubit 1 to qubit 2

In [35]:
ghz_circuit = QuantumCircuit(3)

ghz_circuit.h(0)
ghz_circuit.cx(0, 1)
ghz_circuit.cx(1, 2)

print(ghz_circuit)

     ┌───┐          
q_0: ┤ H ├──■───────
     └───┘┌─┴─┐     
q_1: ─────┤ X ├──■──
          └───┘┌─┴─┐
q_2: ──────────┤ X ├
               └───┘


In [36]:
ghz_state = Statevector.from_instruction(
    ghz_circuit
)

print("GHZ state:")
print(ghz_state)

print("\nGHZ probabilities:")
print(ghz_state.probabilities_dict())

GHZ state:
Statevector([0.70710678+0.j, 0.        +0.j, 0.        +0.j,
             0.        +0.j, 0.        +0.j, 0.        +0.j,
             0.        +0.j, 0.70710678+0.j],
            dims=(2, 2, 2))

GHZ probabilities:
{np.str_('000'): np.float64(0.4999999999999999), np.str_('111'): np.float64(0.4999999999999999)}


## 21. Measuring the GHZ State

The GHZ state contains only:

`|000⟩`

and

`|111⟩`

Therefore measurement should produce only these two
results, approximately 50% each.

In [37]:
ghz_measurement = QuantumCircuit(3, 3)

ghz_measurement.h(0)
ghz_measurement.cx(0, 1)
ghz_measurement.cx(1, 2)

ghz_measurement.measure(
    [0, 1, 2],
    [0, 1, 2]
)

ghz_result = simulator.run(
    ghz_measurement,
    shots=1024
).result()

ghz_counts = ghz_result.get_counts()

print("GHZ measurement:")
print(ghz_counts)

GHZ measurement:
{'000': 514, '111': 510}


In [38]:
plot_histogram(ghz_counts)
plt.show()

# 22. Final Multi-Qubit Experiment

Let's create a Bell state and verify its probabilities.

### Experiment Ideas

Try:

- Remove the Hadamard gate
- Remove the CNOT
- Add an X gate
- Add a Z gate
- Create a different Bell state

The goal is to **experiment and observe**, not just run code.

In [39]:
final_experiment = QuantumCircuit(2)

final_experiment.h(0)
final_experiment.cx(0, 1)

final_state = Statevector.from_instruction(
    final_experiment
)

print("Final circuit:")
print(final_experiment)

print("\nFinal state:")
print(final_state)

print("\nFinal probabilities:")
print(final_state.probabilities_dict())

Final circuit:
     ┌───┐     
q_0: ┤ H ├──■──
     └───┘┌─┴─┐
q_1: ─────┤ X ├
          └───┘

Final state:
Statevector([0.70710678+0.j, 0.        +0.j, 0.        +0.j,
             0.70710678+0.j],
            dims=(2, 2))

Final probabilities:
{np.str_('00'): np.float64(0.4999999999999999), np.str_('11'): np.float64(0.4999999999999999)}


# Module 03 Summary

In this module, we learned:

- Multi-qubit systems
- Computational basis states
- Tensor products
- Multi-qubit superposition
- Measurement
- CNOT gates
- Bell states
- Entanglement
- Density matrices
- Partial trace
- Reduced quantum states
- Separable vs entangled states
- Three-qubit systems
- GHZ states

## Key Idea

For `n` qubits:

`Number of basis states = 2ⁿ`

Quantum systems can also contain correlations and
entanglement that cannot be described simply as
independent classical states.

---
